# Exploration

Let's explore the interactions of all attributes and target variable to help discover drivers of our target variable. 

In [ ]:
#standard ds imports
import pandas as pd
import numpy as np

#viz and stats
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

#my wrangle file
import wrangle

## Acquire and Prepare Data

We're going to acquire the student grade data like we saw in the prior lesson.

In [ ]:
# Use our wrangle function to acquire and prepare our data.
df = wrangle.wrangle_grades()

In [ ]:
#look at your data!
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
# verify student_id is unique
df.student_id.nunique()

In [ ]:
df.final_grade.value_counts()

#### target variable?
- final grade!
- continious aka regression problem

#### other variables of value?
- exam1, exam2, exam3
- all continious

In [ ]:
plt.figure(figsize=(16, 3))

# List of columns
cols = ['exam1', 'exam2', 'exam3', 'final_grade']
for i, col in enumerate(cols):
    # i starts at 0, but plot nos should start at 1
    subplot_num = i+1
    # Create subplot.
    plt.subplot(1,4,subplot_num)
    # Title with column name.
    plt.title(col)
    # Display histogram for column.
    df[col].hist(bins=5)
    # Hide gridlines.
    plt.grid(False)

### What must we do before exploring relationships? 

In [ ]:
#train, validate, test split
train, validate, test = wrangle.split_data(df)

## Explore

### Why Explore?
 - What is the purpose of this pipeline stage?

### Main Stages in Exploration
- Hypothesize
- Visualize
    - Plot out the distributions of each feature 
     - Why?
    - Plot out the interaction of two or more variables?
     - Why?
    - Plot out how subgroups compare to each-other and to the overall population?
     - Why?
    - Document takeaways
     - Why?
    - Identify features that correlate with each other
     - Why?

- Test Hypotheses

### Types of Visualizations

- **Univariate Distributions**
    
    - Continuous variable distributions
        - histogram
        - boxplot
        - displot
    - Discrete variable distributions
        - countplot


- **Bi- and multi-variate relationships** 

    - Continuous with Continuous
        - scatter
        - line
        - pairplot
        - heatmap
        - relplot
    - Discrete with Continuous
        - violin
        - catplot
        - sunburst
        - boxplot
        - swarmplot
        - striplot
        
    - Discrete with Discrete
        - heatmap

___

## Explore our exam dataset
### Goal

> I'm a university professor hoping I can build a prediction model that will be able to use these exams to predict the final grade within 5 points average per student.

## Q1. Is there a relationship between exam1 and final grade? 

- both of my variables are continious
- check for correlation

### Hypothesis

- ${H_0}$: there is no there linear correlation between exam1 and final_grade
- ${H_a}$:  there is a linear correlation between exam1 and final_grade

### Visualize

### `sns.heatmap()`

Let's look at a heatmap of the correlation coefficients for a dataset.

1. Determine if normally distributed
    - if normal, use pearsons method
    - if not, use spearman
2. Calculate the correlation coefficient for each pair of variables
    - use pandas `.corr()` 
    - it defaults to `method=pearson`
    - can change to `method=spearman`
3. Use correleation coefficients to generate heatmap 


In [ ]:
# create the correlation matrix using pandas .corr()
exam_corr = train.drop(columns='student_id').corr(method='spearman')
exam_corr

In [ ]:
# pass my correlation matrix to Seaborn's heatmap
sns.heatmap(exam_corr)
plt.show()

In [ ]:
np.triu(exam_corr)

In [ ]:
# pass my correlation matrix to Seaborn's heatmap with customization
sns.heatmap(exam_corr, cmap='Purples', annot=True, 
            mask=np.triu(exam_corr))
plt.show()

In [ ]:
# pass my correlation matrix to Seaborn's heatmap with more customization! 
kwargs = {'alpha':.9,
          'linewidth':3, 
          'linestyle':'-',
          'linecolor':'black'}

sns.heatmap(exam_corr, cmap='Purples', annot=True, 
            mask=np.triu(exam_corr), **kwargs)
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
X = train[['exam1', 'exam2', 'exam3']]
# VIF dataframe
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
  
# calculating VIF for each feature
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                          for i in range(len(X.columns))]
  
vif_data

### Test Hypothesis

In [ ]:
# Since my variables are not normally distributed, 
# use scipy stats function spearmanr to calculate correlation and p-value 
r, p_value = spearmanr(train.exam1, train.final_grade)

In [ ]:
print(f'Correlation Coefficient by Spearman Test: {r}\np-value: {p_value}')

In [ ]:
# set alpha : 0.05
if p_value < 0.05:
    print('We can reject the null hypothesis')
else:
    print('we cannot reject the null hypothesis')

#### Heatmap Takeaways

- Although all of the exams have very high positive correlations with the target variable, `exam1` and `final_grade` are almost perfectly correlated. This looks to be the best predictor of our target variable.
- Based on my correlation coefficient and my p-value, **I reject my Null hypothesis that there is no correlation between `exam1` and `final_grade`.**
- Looking at the correlation between our independent variables, they also have high positive correlations with each other, multicollinearity. This informs me that I don't want to use all of them together in a linear regression model. 
- When building my model, I will choose `exam1` and perform a simple linear regression first. If I want to go back and do some feature engineering with my other independent variables, I could do that and see if I can improve on my accuracy.

### What other visualizations could we have used?

### `sns.relplot()`

Let's do a simple scatter plot of two continuous variables in our dataset.

In [ ]:
sns.relplot(data=train, x='exam1', y='final_grade')
plt.show()

### `sns.lmplot()`

Let's make that simple scatter plot but add a regression line.

In [ ]:
# I can really pop that line color if I want
sns.lmplot(data=train, x='exam1', y='final_grade',
           line_kws={'color':'red'})
plt.show()

### `sns.jointplot()`

Let's use a `sns.jointplot()` with `kind=reg` to view individual variable distributions for our x and y along with a scatter plot with regression line.

In [ ]:
sns.jointplot(data=train, x='exam1', y='final_grade',
             kind='reg')
plt.show()

### `sns.pairplot()`

Let's use `sns.pairplot()` to view a scatter plot visualizing the relationships between all of the numeric columns in our dataset all at once as well as individual distributions for each individual column.

In [ ]:
# We can drop the redundant information in the upper right half of the chart if we like.
sns.pairplot(data=train.drop(columns='student_id'),
            corner=True)
plt.show()

Takeways
- My `relplot`, `lmplot`, and `jointplots` charts show me that there is a linear relationship between `exam1` and `final_grade`.
- My `pairplot` shows me that there are linear relationships between the other exams scores, as well 

___

## Q2: Is there a cutoff in grade that makes sense to investigate? Passing/failing/letter grades?

In [ ]:
#number of people who failed each test
for col in train.columns[1:]:
    print(f'On {col}, {(train[col] < 70).sum()} students failed.')

### Make categorical values for further exploration

In [ ]:
#assign fail and pass for each test
for col in train.columns[1:]:
    train[col + '_pass'] = np.where( (train[col] < 70), 'fail', 'pass')

In [ ]:
train.head()

### Whats the relationship between passing exam1 and the final_exam grade?

In [ ]:
#seaborn histplot 
sns.histplot(data=train, x='final_grade', 
             hue='exam1_pass', bins=10)

#set line for passing level
plt.vlines(70, 0, 12)
plt.show()

In [ ]:
#mean final_grade by exam1 pass/fail status
train.groupby('exam1_pass').final_grade.mean()

### What percentage of students failed exam1 and the final?

In [ ]:
((train.exam1_pass == 'fail') & (train.final_grade_pass == 'fail')).mean()

### Of the students who failed exam1, how many also failed the final?

In [ ]:
#create subset of people who failed exam1
subset = train [train.exam1_pass == 'fail']
subset

In [ ]:
#how many failed final
(subset.final_grade_pass =='fail').sum()

In [ ]:
#percentage who failed final
(subset.final_grade_pass =='fail').mean()

### Of the students who failed exam2, how many also failed the final?

In [ ]:
#create subset of people who failed exam2
subset = train [train.exam2_pass == 'fail']
subset

In [ ]:
#how many failed final
(subset.final_grade_pass =='fail').sum()

In [ ]:
#percentage who failed final
(subset.final_grade_pass =='fail').mean()

### Of the students who failed both exam1 and exam2, how many also failed the final?

In [ ]:
#create subset of people who failed exam2
subset = train [(train.exam2_pass == 'fail') 
                & (train.exam1_pass == 'fail')]
subset

In [ ]:
#how many failed final
(subset.final_grade_pass =='fail').sum()

In [ ]:
#percentage who failed final
(subset.final_grade_pass =='fail').mean()

#### Takeways
- exam1 and exam2 have a better reflection if someone passes the final exam
- given the multicollinearity present, we may want to just focus on one exam
- if i had time, we could create a new feature that captures if a student failed both exam1 and exam2 